In [ ]:
import pandas as pd

from Pipeline.Global.GlobalSetting import GlobalSetting

In [1]:
models_to_evaluate = ["RELM", "RELM_CV", "RELM_CV_Ensemble"]
config_type = "Grid_Optimization"

abc_results_list = []

for model_name in models_to_evaluate:
    for fold_id in range(5):
        try:
            # Construct the dynamic filename matching the new ABC_Testing.py export format
            # Note: Do not add .csv if get_dataframe_from_record handles the extension internally
            file_name = f"{config_type}_ABC_{model_name}_Fold_{fold_id}_Results"

            # Read the CSV generated by the parallel notebooks
            df_fold = GlobalSetting.get_dataframe_from_record(file_name)

            # --- 3. Inject Contextual Metadata ---
            # This allows you to group-by model and fold later during plotting
            df_fold["Model_Type"] = model_name
            df_fold["Fold_ID"] = fold_id

            abc_results_list.append(df_fold)
            print(f"[Success] Extracted: {model_name} - Fold {fold_id}")

        except Exception as e:
            # Failsafe for distributed synchronization delays
            print(f"[WARNING] Missing or corrupt data for {model_name} - Fold {fold_id}. Error: {e}")

# --- 4. Final Data Matrix Construction ---
if abc_results_list:
    # ignore_index=True ensures the final dataframe has a clean, continuous index
    final_combined_df = pd.concat(abc_results_list, ignore_index=True)

    # Optional: Save the master dataframe so you don't have to re-aggregate
    # GlobalSetting.save_dataframe_to_record(final_combined_df, "Master_Comparative_Results")

    display(final_combined_df.head())
else:
    print("[ERROR] No result files found. Verify the Storage/Record directory path.")

[I/O Trace] Record imported successfully: ../../Storage/Record\ABC_Grid_Optimization_Fold_0_Results.csv
[I/O Trace] Record imported successfully: ../../Storage/Record\ABC_Grid_Optimization_Fold_1_Results.csv
[I/O Trace] Record imported successfully: ../../Storage/Record\ABC_Grid_Optimization_Fold_2_Results.csv
[I/O Trace] Record imported successfully: ../../Storage/Record\ABC_Grid_Optimization_Fold_3_Results.csv
[I/O Trace] Record imported successfully: ../../Storage/Record\ABC_Grid_Optimization_Fold_4_Results.csv
Loaded ABC Data Shape: (150, 15) | Expected: (150, X)


In [2]:
# --- 3. Aggregate across ALL folds and seeds to match the baseline ---

# We remove 'Fold_ID' from the group_cols so it collapses all 5 folds together
group_cols = ["Model_Type", "Hidden_Nodes", "Lambda_Value", "Activation"]

# Dynamically extract metrics, explicitly ignoring BOTH "Seed" and "Fold_ID"
metric_cols = [col for col in df_abc_master_raw.columns if col not in group_cols + ["Seed", "Fold_ID"]]

# Calculate mean, std, sem, max, min across all 150 data points
agg_funcs = {metric: ['mean', 'std', 'sem', 'max', 'min'] for metric in metric_cols}

df_abc_summary = df_abc_master_raw.groupby(group_cols).agg(agg_funcs).reset_index()

# Flatten Multi-Index Columns
df_abc_summary.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in df_abc_summary.columns.values
]

df_abc_summary

,Model_Type,Hidden_Nodes,Lambda_Value,Activation,Accuracy_mean,Accuracy_std,Accuracy_sem,Accuracy_max,Accuracy_min,Precision_mean,...,Bal Accuracy_mean,Bal Accuracy_std,Bal Accuracy_sem,Bal Accuracy_max,Bal Accuracy_min,MCC_mean,MCC_std,MCC_sem,MCC_max,MCC_min
0,ABC_Optimized_ELM,37,0.077887,sigmoid,0.754565,0.056491,0.004612,0.888889,0.640625,0.796231,...,0.753782,0.056586,0.00462,0.889113,0.640625,0.516966,0.111583,0.009111,0.778226,0.282494


In [3]:
# Load your standard ELM baseline results using the safe loader
baseline_file = "Model_Testing_Result"
df_baseline_summary = GlobalSetting.get_dataframe_from_record(baseline_file)

# --- THE GRAND MERGE ---
# Stack the ABC results directly underneath the Standard ELM results
final_comparison_df = pd.concat([df_baseline_summary, df_abc_summary], ignore_index=True)

# Sort the table by Model_Type so you can easily compare them
final_comparison_df = final_comparison_df.sort_values(by=["Model_Type"]).reset_index(drop=True)

# Save the ultimate FYP result table
GlobalSetting.save_dataframe_to_record(final_comparison_df, "ULTIMATE_FYP_Comparison.csv")

# Force Jupyter to display ALL columns without truncating them
pd.set_option('display.max_columns', None)

# Show the entire dataframe
final_comparison_df

[I/O Trace] Record imported successfully: ../../Storage/Record\Model_Testing_Result.csv
[I/O Trace] Record exported successfully: ../../Storage/Record\ULTIMATE_FYP_Comparison.csv


,Model_Type,Hidden_Nodes,Lambda_Value,Activation,Accuracy_mean,Accuracy_sem,Accuracy_std,Accuracy_max,Accuracy_min,Precision_mean,Precision_sem,Precision_std,Precision_max,Precision_min,Recall_mean,Recall_sem,Recall_std,Recall_max,Recall_min,NPV_mean,NPV_sem,NPV_std,NPV_max,NPV_min,Specificity_mean,Specificity_sem,Specificity_std,Specificity_max,Specificity_min,F1-Score_mean,F1-Score_sem,F1-Score_std,F1-Score_max,F1-Score_min,F2-Score_mean,F2-Score_sem,F2-Score_std,F2-Score_max,F2-Score_min,Bal Accuracy_mean,Bal Accuracy_sem,Bal Accuracy_std,Bal Accuracy_max,Bal Accuracy_min,MCC_mean,MCC_sem,MCC_std,MCC_max,MCC_min
0,ABC_Optimized_ELM,37,0.077887,sigmoid,0.754565,0.004612,0.056491,0.888889,0.640625,0.796231,0.005038,0.061705,0.947368,0.655172,0.681586,0.008237,0.100881,0.90625,0.516129,0.730545,0.005494,0.067282,0.903226,0.615385,0.825979,0.005154,0.063121,0.968750,0.68750,0.730523,0.005674,0.069487,0.888889,0.596491,0.699613,0.007204,0.088234,0.897436,0.547945,0.753782,0.004620,0.056586,0.889113,0.640625,0.516966,0.009111,0.111583,0.778226,0.282494
1,Best_Hidden_Nodes,48,0.000000,sigmoid,0.752194,0.004521,0.055374,0.857143,0.609375,0.787264,0.004889,0.059872,0.954545,0.629630,0.688266,0.008390,0.102758,0.93750,0.437500,0.732628,0.005513,0.067526,0.923077,0.589744,0.814855,0.005135,0.062888,0.968750,0.68750,0.730341,0.005701,0.069823,0.857143,0.561404,0.703717,0.007298,0.089376,0.903614,0.486111,0.751560,0.004519,0.055345,0.856855,0.609375,0.511332,0.008914,0.109172,0.717052,0.221470
2,Best_Lambda,48,0.031250,sigmoid,0.763839,0.004415,0.054078,0.888889,0.640625,0.808238,0.005247,0.064265,0.958333,0.680000,0.691465,0.008190,0.100309,0.90625,0.500000,0.739057,0.005334,0.065329,0.903226,0.615385,0.834684,0.005501,0.067367,0.969697,0.65625,0.740841,0.005466,0.066949,0.888889,0.592593,0.709559,0.007073,0.086624,0.897436,0.533333,0.763075,0.004411,0.054028,0.889113,0.640625,0.536489,0.008720,0.106801,0.778226,0.288231
3,Best_Size_and_Lambda,48,0.031250,sigmoid,0.763839,0.004415,0.054078,0.888889,0.640625,0.808238,0.005247,0.064265,0.958333,0.680000,0.691465,0.008190,0.100309,0.90625,0.500000,0.739057,0.005334,0.065329,0.903226,0.615385,0.834684,0.005501,0.067367,0.969697,0.65625,0.740841,0.005466,0.066949,0.888889,0.592593,0.709559,0.007073,0.086624,0.897436,0.533333,0.763075,0.004411,0.054028,0.889113,0.640625,0.536489,0.008720,0.106801,0.778226,0.288231
4,Grid_Combination,93,0.062500,sigmoid,0.766782,0.004633,0.056745,0.888889,0.656250,0.811568,0.004900,0.060012,0.950000,0.692308,0.691909,0.008233,0.100834,0.87500,0.500000,0.740470,0.005501,0.067379,0.878788,0.625000,0.840057,0.004890,0.059889,0.969697,0.68750,0.743260,0.005720,0.070058,0.885246,0.607143,0.710865,0.007222,0.088450,0.876623,0.544218,0.765983,0.004629,0.056691,0.888609,0.656250,0.541806,0.009080,0.111212,0.780948,0.322749
5,Grid_Optimization,37,0.077887,sigmoid,0.756438,0.004376,0.053600,0.904762,0.656250,0.797755,0.005170,0.063314,0.952381,0.678571,0.685618,0.007575,0.092776,0.87500,0.531250,0.732072,0.004987,0.061075,0.882353,0.631579,0.825821,0.005447,0.066709,0.968750,0.68750,0.733812,0.005294,0.064844,0.900000,0.618182,0.703423,0.006624,0.081132,0.882353,0.562914,0.755719,0.004381,0.053661,0.904234,0.656250,0.520452,0.008680,0.106308,0.810924,0.314970
